# dataset_build

> Turn a folder of full-resolution source images into a patch dataset with a reproducible train/val split and a metadata manifest.

The split happens at the **source** level (not the patch level): all patches from one source image go entirely to train or entirely to val. Splitting at the patch level would leak near-identical overlapping crops across the boundary and inflate validation scores.

In [ ]:
#| default_exp dataset_build

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import random
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple


In [ ]:
#| exporti
from segmentation_tools_pt.patches import PatchConfig, extract_patches_from_sample

## Discover source samples

Supports two on-disk layouts:

| Layout | `n_channels` | Images | Masks |
|--------|-------------|--------|-------|
| Single-channel | `1` | `images/{id}.png` | `masks/{id}.png` or `masks/{id}_mask.png` |
| Multi-channel | `≥2` | `images/{id}_ch0.png`, `{id}_ch1.png`, … | `masks/{id}_mask.png` or `masks/{id}.png` |

Also accepts `{id}_image1.png`, `{id}_image2.png` (legacy pin-detection naming).

Sample ids are deduplicated: `foo_ch0.png` and `foo_ch1.png` → one sample `foo`.

In [ ]:
#| export
# Suffixes stripped from image stems when grouping multi-channel files into one sample.
_CHANNEL_SUFFIXES = ('_ch0', '_ch1', '_ch2', '_ch3', '_image1', '_image2', '_image3', '_image4')

def _stem_to_sample_id(stem: str, n_channels: int) -> str:
    """Map an image filename stem to a sample id."""
    if n_channels == 1:
        return stem
    for suf in _CHANNEL_SUFFIXES:
        if stem.endswith(suf):
            return stem[: -len(suf)]
    return stem


In [ ]:
#| export
def resolve_mask_path(
    source_dir: Path,
    sample_id: str,
    exts: str = '.png',
    mask_suffix: str = '_mask',
) -> Path:
    """Return the mask file for `sample_id`, trying common naming patterns."""
    masks = source_dir / 'masks'
    candidates = []
    if mask_suffix:
        candidates.append(masks / f'{sample_id}{mask_suffix}{exts}')
    candidates.append(masks / f'{sample_id}{exts}')
    if mask_suffix != '_mask':
        candidates.append(masks / f'{sample_id}_mask{exts}')
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f'No mask for {sample_id!r} in {masks} (tried {[c.name for c in candidates]})')


In [ ]:
#| export
def resolve_mask_paths(
    source_dir: Path,
    sample_id: str,
    class_names: Sequence[str],
    exts: str = '.png',
) -> List[Path]:
    """Return the K per-class mask files for `sample_id` (`mask_mode='per_class_files'`)."""
    masks = source_dir / 'masks'
    paths: List[Path] = []
    for name in class_names:
        p = masks / f'{sample_id}_{name}{exts}'
        if not p.exists():
            raise FileNotFoundError(f'Missing mask channel {name!r} for {sample_id!r}: {p}')
        paths.append(p)
    return paths


In [ ]:
#| export
def resolve_channel_paths(
    source_dir: Path,
    sample_id: str,
    n_channels: int,
    exts: str = '.png',
) -> List[Path]:
    """Return image paths for each source channel of `sample_id`."""
    images = source_dir / 'images'
    if n_channels == 1:
        p = images / f'{sample_id}{exts}'
        if not p.exists():
            raise FileNotFoundError(f'Missing image: {p}')
        return [p]
    paths: List[Path] = []
    for i in range(n_channels):
        candidates = [
            images / f'{sample_id}_ch{i}{exts}',
            images / f'{sample_id}_image{i + 1}{exts}',
        ]
        found = next((c for c in candidates if c.exists()), None)
        if found is None:
            raise FileNotFoundError(
                f'Missing channel {i} for {sample_id!r} in {images} '
                f'(expected {sample_id}_ch{i}{exts} or {sample_id}_image{i + 1}{exts})'
            )
        paths.append(found)
    return paths


In [ ]:
#| export
def discover_samples(
    source_dir: str,
    n_channels: int = 1,
    exts: str = '.png',
    mask_suffix: str = '_mask',
    mask_mode: str = 'single',                    # 'single' | 'stacked' | 'per_class_files'
    class_names: Optional[Sequence[str]] = None,   # required for mask_mode='per_class_files'
) -> List[str]:
    """List sample ids that have a mask and the expected channel images.

    Example:
        >>> # single-channel: images/a.png + masks/a.png
        >>> discover_samples('/data', n_channels=1)
        ['a']
    """
    assert mask_mode in ('single', 'stacked', 'per_class_files'), f'unknown mask_mode {mask_mode!r}'
    if mask_mode == 'per_class_files':
        assert class_names is not None, "mask_mode='per_class_files' requires class_names"
    src = Path(source_dir)
    ids: set = set()
    for p in (src / 'images').glob(f'*{exts}'):
        ids.add(_stem_to_sample_id(p.name[: -len(exts)], n_channels))
    if mask_mode != 'per_class_files':
        # 'single'/'stacked' masks are named after the sample id directly; per-class
        # files (`{id}_{class}.png`) aren't 1:1 with the id, so skip this pass for them
        # and rely on ids already collected from images/ instead.
        for p in (src / 'masks').glob(f'*{exts}'):
            stem = p.name[: -len(exts)]
            if stem.endswith('_mask'):
                stem = stem[: -len('_mask')]
            ids.add(stem)
    valid = []
    for sid in sorted(ids):
        try:
            resolve_channel_paths(src, sid, n_channels, exts)
            if mask_mode == 'per_class_files':
                resolve_mask_paths(src, sid, class_names, exts)
            else:
                resolve_mask_path(src, sid, exts, mask_suffix)
            valid.append(sid)
        except FileNotFoundError:
            continue
    if not valid:
        raise FileNotFoundError(f'No samples with matching images+masks in {src}')
    return valid

In [ ]:
#| eval: false
RAW_DIR=Path(r'/path/to/data/example_line/segmentation/00000')
im_dir = Path(RAW_DIR, 'images')
mask_dir = Path(RAW_DIR, 'masks')

In [ ]:
#| eval: false
sample_ids = discover_samples(RAW_DIR)

## Reproducible source-level split

In [ ]:
# experiment
ids = [f's{i}' for i in range(10)]
r = random.Random(42); r.shuffle(ids)
n_val = max(1, int(len(ids) * 0.2))
print('val:', ids[:n_val], '| train:', ids[n_val:])

In [ ]:
#| export
def split_sources(
    sample_ids: List[str],   # all source ids
    val_fraction: float,     # fraction held out for validation
    seed: int,               # shuffle seed
) -> Tuple[List[str], List[str]]:
    """Shuffle and split ids into `(train_ids, val_ids)` (lists stay ordered).

    Example:
        >>> tr, va = split_sources([f's{i}' for i in range(10)], 0.2, 42)
        >>> len(va), len(tr)
        (2, 8)
    """
    ids = list(sample_ids)
    random.Random(seed).shuffle(ids)
    n_val = max(1, int(len(ids) * val_fraction))
    return ids[n_val:], ids[:n_val]

In [ ]:
#| eval: false
tr, va = split_sources(sample_ids, 0.2, 42)

In [ ]:
#| eval: false
tr, va

In [ ]:
# test
tr, va = split_sources([f's{i}' for i in range(10)], 0.2, 42)
assert len(va) == 2 and len(tr) == 8 and set(tr).isdisjoint(va)

## Parallel build

The worker is defined at module level so it pickles cleanly under the `spawn`/`forkserver` start methods (not just `fork`). `build_patch_dataset` fans the sources out across processes, then writes `train.txt`, `val.txt` and `metadata.json`.

In [ ]:
#| export
def _process_sample(
    sid: str, split: str, source_dir: str, output_dir: str,
    cfg: PatchConfig, num_classes: int, n_channels: int, fit_rects: bool,
    exts: str = '.png', mask_suffix: str = '_mask',
    mask_mode: str = 'single', mask_channels: Optional[int] = None,
    class_names: Optional[Sequence[str]] = None,
) -> Tuple[str, List[Dict]]:
    "Top-level worker for `build_patch_dataset` (must be importable for pickling)."
    src = Path(source_dir)
    channel_paths = resolve_channel_paths(src, sid, n_channels, exts)
    if mask_mode == 'per_class_files':
        mask_path = None
        mask_paths = [str(p) for p in resolve_mask_paths(src, sid, class_names, exts)]
    else:
        mask_path = str(resolve_mask_path(src, sid, exts, mask_suffix))
        mask_paths = None
    meta = extract_patches_from_sample(
        sample_id=sid,
        channel_paths=[str(p) for p in channel_paths],
        mask_path=mask_path,
        output_dir=Path(output_dir) / split, cfg=cfg,
        num_classes=num_classes, fit_rects=fit_rects,
        mask_mode=mask_mode, mask_channels=mask_channels,
        mask_paths=mask_paths, class_names=class_names,
    )
    return split, meta


In [ ]:
PatchConfig?

In [ ]:
cfg = PatchConfig(
    patch_size=256, 
    stride=128,
    min_fg_pixels=1,
    max_empty_fraction=0.0,
    val_fraction=0.2,
    seed=42,
)



In [ ]:
#| export
def build_patch_dataset(
    source_dir: str,
    output_dir: str,
    cfg: PatchConfig = PatchConfig(),
    num_classes: int = 1,
    n_channels: int = 1,
    n_jobs: int = 4,
    fit_rects: bool = False,
    image_exts: str = '.png',
    mask_suffix: str = '_mask',
    mask_mode: str = 'single',                    # 'single' | 'stacked' | 'per_class_files'
    mask_channels: Optional[int] = None,           # K, required for 'stacked'
    class_names: Optional[Sequence[str]] = None,   # K class name suffixes, required for 'per_class_files'
) -> Dict[str, List[Dict]]:
    """Build a patch dataset from `source_dir` into `output_dir`.

    Set `n_channels=1` when images are plain `{id}.png` (no `_ch0` suffix).
    Set `n_channels=2` for `{id}_ch0.png` / `{id}_ch1.png` or `{id}_image1.png` / `{id}_image2.png`.
    Masks: `{id}_mask.png` or `{id}.png` (pass `mask_suffix=''` if masks never use `_mask`), unless
    `mask_mode='per_class_files'` (`{id}_{class_name}.png` per class) or `mask_mode='stacked'`
    (one `mask_channels`-channel PNG, same naming as `single`).
    """
    ids = discover_samples(source_dir, n_channels=n_channels, exts=image_exts, mask_suffix=mask_suffix,
                            mask_mode=mask_mode, class_names=class_names)
    train_ids, val_ids = split_sources(ids, cfg.val_fraction, cfg.seed)
    jobs = [(s, 'train') for s in train_ids] + [(s, 'val') for s in val_ids]
    all_meta: Dict[str, List[Dict]] = {'train': [], 'val': []}

    def run(sid, split):
        return _process_sample(
            sid, split, source_dir, output_dir, cfg, num_classes, n_channels, fit_rects,
            exts=image_exts, mask_suffix=mask_suffix,
            mask_mode=mask_mode, mask_channels=mask_channels, class_names=class_names,
        )

    if n_jobs > 1:
        with ProcessPoolExecutor(max_workers=n_jobs) as ex:
            futs = {
                ex.submit(
                    _process_sample, s, sp, source_dir, output_dir, cfg,
                    num_classes, n_channels, fit_rects, image_exts, mask_suffix,
                    mask_mode, mask_channels, class_names,
                ): (s, sp)
                for s, sp in jobs
            }
            for fut in as_completed(futs):
                split, meta = fut.result()
                all_meta[split].extend(meta)
    else:
        for sid, split in jobs:
            split, meta = run(sid, split)
            all_meta[split].extend(meta)

    out = Path(output_dir)
    for split in ('train', 'val'):
        (out / f'{split}.txt').write_text('\n'.join(m['sample_id'] for m in all_meta[split]))
    (out / 'metadata.json').write_text(json.dumps(all_meta, indent=2))
    return all_meta

In [ ]:
#| eval: false
OUT_DIR = Path(r'/path/to/data/example_line/segmentation/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#| eval: false
all_meta = build_patch_dataset(
    RAW_DIR, OUT_DIR,
    num_classes=1,
    n_channels=1,
    n_jobs=4,
    fit_rects=False,
    image_exts='.png',
    mask_suffix='_mask',
)

In [ ]:
#| eval: false
all_meta

In [ ]:
# test: single-channel layout (plain filenames) and multi-channel layout (_ch0/_ch1)
import tempfile
import cv2
import numpy as np
rng = np.random.default_rng(0)

with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    # --- single-channel (plain names, no _ch0 suffix) ---
    (d / 'single' / 'images').mkdir(parents=True)
    (d / 'single' / 'masks').mkdir(parents=True)
    for name in ('a', 'b'):
        cv2.imwrite(str(d / 'single' / 'images' / f'{name}.png'), rng.integers(0, 256, (60, 60), np.uint8))
        m = np.zeros((60, 60), np.uint8); m[20:40, 20:40] = 255
        cv2.imwrite(str(d / 'single' / 'masks' / f'{name}.png'), m)
    assert discover_samples(str(d / 'single'), n_channels=1) == ['a', 'b']
    cfg = PatchConfig(patch_size=20, stride=20, min_fg_pixels=1, max_empty_fraction=0.0, val_fraction=0.5)
    meta1 = build_patch_dataset(str(d / 'single'), str(d / 'out_single'), cfg=cfg, n_channels=1, n_jobs=1, mask_suffix='')
    assert meta1['train'] and meta1['val']

    # --- multi-channel ---
    (d / 'multi' / 'images').mkdir(parents=True)
    (d / 'multi' / 'masks').mkdir(parents=True)
    for i in range(2):
        for c in range(2):
            cv2.imwrite(str(d / 'multi' / 'images' / f's{i}_ch{c}.png'), rng.integers(0, 256, (60, 60), np.uint8))
        m = np.zeros((60, 60), np.uint8); m[20:40, 20:40] = 255
        cv2.imwrite(str(d / 'multi' / 'masks' / f's{i}_mask.png'), m)
    assert discover_samples(str(d / 'multi'), n_channels=2) == ['s0', 's1']
    meta2 = build_patch_dataset(str(d / 'multi'), str(d / 'out_multi'), cfg=cfg, n_channels=2, n_jobs=1)
    assert meta2['train'] and (d / 'out_multi' / 'metadata.json').exists()

In [ ]:
# test: multilabel end-to-end build -- 'stacked' and 'per_class_files' through discover_samples + build_patch_dataset
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    (d / 'stacked' / 'images').mkdir(parents=True)
    (d / 'stacked' / 'masks').mkdir(parents=True)
    stacked = np.zeros((60, 60, 3), np.uint8)
    stacked[10:50, 10:50, 0] = 255
    stacked[25:35, 25:35, 1] = 255
    stacked[15:45, 15:45, 2] = 255
    for name in ('a', 'b'):
        cv2.imwrite(str(d / 'stacked' / 'images' / f'{name}.png'), rng.integers(0, 256, (60, 60), np.uint8))
        cv2.imwrite(str(d / 'stacked' / 'masks' / f'{name}.png'), stacked)
    assert discover_samples(str(d / 'stacked'), n_channels=1, mask_mode='stacked') == ['a', 'b']
    cfg = PatchConfig(patch_size=20, stride=20, min_fg_pixels=1, max_empty_fraction=0.0, val_fraction=0.5)
    meta_s = build_patch_dataset(str(d / 'stacked'), str(d / 'out_stacked'), cfg=cfg, n_channels=1, n_jobs=1,
                                  mask_mode='stacked', mask_channels=3)
    assert meta_s['train'] and meta_s['val']
    sample_mask = cv2.imread(str(d / 'out_stacked' / 'train' / 'masks' / f"{meta_s['train'][0]['sample_id']}.png"),
                              cv2.IMREAD_UNCHANGED)
    assert sample_mask.shape == (20, 20, 3)

    (d / 'pcf' / 'images').mkdir(parents=True)
    (d / 'pcf' / 'masks').mkdir(parents=True)
    chip = np.zeros((60, 60), np.uint8); chip[10:50, 10:50] = 255
    void = np.zeros((60, 60), np.uint8); void[25:35, 25:35] = 255
    lifted = np.zeros((60, 60), np.uint8); lifted[15:45, 15:45] = 255
    for name in ('a', 'b'):
        cv2.imwrite(str(d / 'pcf' / 'images' / f'{name}.png'), rng.integers(0, 256, (60, 60), np.uint8))
        cv2.imwrite(str(d / 'pcf' / 'masks' / f'{name}_chip.png'), chip)
        cv2.imwrite(str(d / 'pcf' / 'masks' / f'{name}_void.png'), void)
        cv2.imwrite(str(d / 'pcf' / 'masks' / f'{name}_lifted.png'), lifted)
    class_names = ['chip', 'void', 'lifted']
    assert discover_samples(str(d / 'pcf'), n_channels=1, mask_mode='per_class_files', class_names=class_names) == ['a', 'b']
    meta_p = build_patch_dataset(str(d / 'pcf'), str(d / 'out_pcf'), cfg=cfg, n_channels=1, n_jobs=1,
                                  mask_mode='per_class_files', class_names=class_names)
    assert meta_p['train'] and meta_p['val']
    pid = meta_p['train'][0]['sample_id']
    for name in class_names:
        assert (d / 'out_pcf' / 'train' / 'masks' / f'{pid}_{name}.png').exists()
print('build_patch_dataset ✓  multilabel end-to-end: discover_samples + build for stacked and per_class_files')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()